# 07 時間序列與預測 — 參考解答

松柏護理之家退伍軍人症群聚事件時間序列練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.metrics import mean_absolute_error

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["hospitalization_date"] = pd.to_datetime(df["hospitalization_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
cases = df[df["infected"] == 1]

## 題目 1：建立每日住院數序列

In [ ]:
# 每日住院數
hosp_cases = cases[cases["hospitalization_date"].notna()]
hosp_daily = hosp_cases.groupby("hospitalization_date").size()
hosp_daily = hosp_daily.asfreq("D", fill_value=0)
hosp_daily.name = "hospitalizations"

print(f"序列長度：{len(hosp_daily)} 天")
print(f"日期範圍：{hosp_daily.index.min().date()} \u2013 {hosp_daily.index.max().date()}")
print(f"住院總數：{hosp_daily.sum()}")

# 住院曲線 + 5 日滾動平均
rolling_5 = hosp_daily.rolling(window=5, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(hosp_daily.index, hosp_daily.values, color="#e34a33", alpha=0.6,
       label="每日住院")
ax.plot(rolling_5.index, rolling_5.values, color="navy", linewidth=2,
        label="5 日滾動平均")
ax.set_title("每日住院曲線 + 5 日滾動平均")
ax.set_xlabel("住院日期")
ax.set_ylabel("住院人數")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 題目 2：住院數預測與窗口比較

In [ ]:
# 窗口比較
print("=== 住院數預測 MAE ===")
best_w, best_mae = 3, float("inf")

for w in [3, 5, 7]:
    pred_w = hosp_daily.rolling(window=w).mean().shift(1).dropna()
    actual_w = hosp_daily.loc[pred_w.index]
    mae_w = mean_absolute_error(actual_w, pred_w)
    print(f"  window={w}  MAE={mae_w:.3f}")
    if mae_w < best_mae:
        best_w, best_mae = w, mae_w

print(f"\n\u2192 最佳窗口：window={best_w}（MAE={best_mae:.3f}）")

# Actual vs Predicted
pred_best = hosp_daily.rolling(window=best_w).mean().shift(1).dropna()
actual_best = hosp_daily.loc[pred_best.index]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(actual_best.index, actual_best.values, marker="o", markersize=4,
        label="實際住院", color="#e34a33")
ax.plot(pred_best.index, pred_best.values, marker="s", markersize=4,
        label=f"預測（{best_w} 日 MA）", color="navy", linestyle="--")
ax.set_title(f"住院數 Actual vs Predicted（{best_w} 日滾動平均）")
ax.set_xlabel("日期")
ax.set_ylabel("每日住院數")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 題目 3（挑戰題）：按嚴重度分組的流行曲線

In [ ]:
# 按嚴重度建立每日發病數序列
severity_levels = ["mild", "moderate", "severe"]
colors = {"mild": "#41b6c4", "moderate": "#fed976", "severe": "#e31a1c"}

# 建立每日序列（所有嚴重度共用日期範圍）
all_onset = cases.groupby("symptom_onset_date").size()
all_onset = all_onset.asfreq("D", fill_value=0)
all_dates = all_onset.index

severity_daily = {}
for sev in severity_levels:
    sub = cases[cases["clinical_severity"] == sev]
    s = sub.groupby("symptom_onset_date").size()
    severity_daily[sev] = s.reindex(all_dates, fill_value=0)

sev_df = pd.DataFrame(severity_daily)

# Stacked bar chart
fig, ax = plt.subplots(figsize=(10, 4))
bottom = np.zeros(len(sev_df))

for sev in severity_levels:
    ax.bar(sev_df.index, sev_df[sev].values, bottom=bottom,
           color=colors[sev], alpha=0.8, label=sev)
    bottom += sev_df[sev].values

ax.set_title("流行曲線（按嚴重度分層）")
ax.set_xlabel("發病日期")
ax.set_ylabel("病例數")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

# 各嚴重度的高峰日
print("=== 各嚴重度高峰日 ===")
for sev in severity_levels:
    peak_date = sev_df[sev].idxmax()
    peak_count = sev_df[sev].max()
    print(f"  {sev:10s}  高峰日 = {peak_date.date()}  當日 {peak_count} 人")

print("\n\u2192 觀察重症病例是否與輕症同步出現，或有時間延遲")
print("\u2192 如果重症集中在疫情中期，可能代表暴露劑量較高的住民較晚發病")

### 解讀

- **住院曲線**：住院高峰比發病高峰晚幾天，這個 lag 可用於預測床位需求
- **窗口選擇**：較小窗口（3 日）通常在急性群聚中表現較好，因為病例數變化快
- **嚴重度分層**：如果重症病例集中在某個時間段，可能提示特定暴露事件或高風險族群
- **限制**：滾動平均是最簡單的 baseline 模型，無法捕捉趨勢轉折點；更進階的方法（如 ARIMA）可在此基礎上改進